# 🚁 Pi0 Drone VLA — Inference Test (Kaggle)
### Menggunakan Checkpoint 2500 Steps

Notebook ini melakukan **pengujian inferensi** terhadap model Pi0 Drone VLA yang telah dilatih.
Pastikan dataset `dronepivla` sudah di-attach dan GPU T4 sudah aktif.

In [ ]:
import os, subprocess, shutil, sys
from huggingface_hub import snapshot_download

# 1. Konfigurasi Nama dan Repo
REPO_ID = 'izunx/drone-roblok-2500'
# Kita gunakan folder lokal di Kaggle agar os.path bisa bekerja
KAGGLE_BASE = '/kaggle/working/dronepivla'

print(f'⏳ Menyiapkan Link ke Hugging Face: {REPO_ID}')

# 2. Proses Sinkronisasi (Download)
# Jika folder belum ada, kita download otomatis
if not os.path.exists(KAGGLE_BASE):
    print(f'📂 Folder lokal tidak ditemukan. Mendownload dari Hub...')
    try:
        snapshot_download(
            repo_id=REPO_ID,
            local_dir=KAGGLE_BASE,
            local_dir_use_symlinks=False
        )
        print('✅ Download selesai.')
    except Exception as e:
        print(f'❌ Gagal mendownload: {e}')
else:
    print('✅ Folder lokal ditemukan, melewati proses download.')

# 3. Definisikan Path Lokal (Sekarang os.path akan bekerja)
OPENPI_DIR     = os.path.join(KAGGLE_BASE, 'openpi')
DATASET_DIR    = os.path.join(KAGGLE_BASE, 'drone_roblox')
NORMSTATS_DIR  = KAGGLE_BASE
CHECKPOINT_DIR = KAGGLE_BASE

OPENPI_ZIP   = os.path.join(KAGGLE_BASE, 'openpi.zip')
DATASET_ZIP  = os.path.join(KAGGLE_BASE, 'drone_roblox.zip')

print('\n🔍 Melakukan Verifikasi File Lokal...')

def check_path(dir_path):
    if dir_path and os.path.exists(dir_path):
        return dir_path
    return None

FINAL_OPENPI       = check_path(OPENPI_DIR)
FINAL_DATASET      = check_path(DATASET_DIR)
FINAL_CHECKPOINT   = check_path(CHECKPOINT_DIR)
MODEL_SAFETENSORS  = os.path.join(CHECKPOINT_DIR, 'model.safetensors')

# Tampilkan Hasil Verifikasi
missing = False
for name, p in [
    ('OpenPI Source', FINAL_OPENPI),
    ('Drone Dataset', FINAL_DATASET),
    ('model.safetensors', MODEL_SAFETENSORS if os.path.exists(MODEL_SAFETENSORS) else None),
    ('norm_stats.json', os.path.join(KAGGLE_BASE, 'norm_stats.json') if os.path.exists(os.path.join(KAGGLE_BASE, 'norm_stats.json')) else None),
]:
    if p:
        print(f'  ✅ {name:20} : Ditemukan')
    else:
        print(f'  ❌ {name:20} : TIDAK DITEMUKAN!')
        missing = True

if not missing:
    print('\n🚀 Semua file siap! Kamu bisa lanjut ke sel pemuatan model.')
else:
    print('\n⚠️ PERINGATAN: Beberapa file masih hilang. Periksa struktur repo di Hugging Face.')


In [ ]:
%%time
# ^-- Tetap di baris pertama tanpa ada komentar di atasnya!

import subprocess, sys

def pip_install(*args):
    """Run pip install with given arguments."""
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-warn-conflicts', *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'⚠️  Warning during install: {result.stderr[-500:]}')
    return result.returncode

# Step 0: Pembersihan Total (Kaggle sering punya library sisa yang bentrok)
print('🧹 Step 0: Nuking existing JAX, Numpy, and Torch to prevent version mismatch...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'jax', 'jaxlib', 'numpy', 'torch', 'torchaudio', 'torchvision', '-y'], capture_output=True)

print('📦 Step 1: Installing Numpy 1.26.4 (CRITICAL: Fixes the umath error)...')
pip_install('numpy==1.26.4')

print('📦 Step 2: Installing Torch & Core ML libraries...')
pip_install(
    'torch==2.4.0', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121',
    'safetensors>=0.4.0',
    'einops>=0.8.0',
    'wandb>=0.19.1',
)

print('📦 Step 3: Installing JAX + Flax ecosystem (0.5.3)...')
pip_install(
    'jax[cuda12]==0.5.3',
    'jaxlib==0.5.3',
    'flax==0.10.2',
    'orbax-checkpoint==0.11.13',
    'chex',
    'equinox>=0.11.8',
    'ml-collections==1.0.0',
    'jaxtyping==0.2.36',
    'ml-dtypes==0.4.1',
)

print('📦 Step 4: Installing data libraries...')
pip_install(
    'lerobot@git+https://github.com/huggingface/lerobot@0cf864870cf29f4738d3ade893e6fd13fbd7cdb5',
    'datasets',
    'huggingface-hub',
)

print('📦 Step 5: Installing all utility libraries (FULL LIST)...')
pip_install(
    'tyro>=0.9.5',
    'beartype==0.19.0',
    'numpydantic>=1.6.6',
    'sentencepiece>=0.2.0',
    'tqdm-loggable>=0.2',
    'humanize',
    'filelock>=3.16.1',
    'treescope>=0.1.7',
    'augmax>=0.3.4',
    'dm-tree>=0.1.8',
    'flatbuffers>=24.3.25',
    'fsspec==2023.10.0',
    'gcsfs',
    'imageio>=2.36.1',
    'opencv-python-headless',
    'pillow>=11.0.0',
    'rich>=14.0.0',
    'polars>=1.30.0',
    'transformers==4.53.2',
    'gym-aloha>=0.1.1',
)

print('\n✅ All dependencies installed!')
print('⚠️  WAJIB: Klik menu "Run" (panel atas Kaggle) -> "Restart Kernel"!')

In [ ]:
import os, shutil, glob, subprocess, sys

# 1. Konfigurasi Path
HOME_DIR = os.path.expanduser('~')
# Folder dataset di cache HF (untuk policy loader)
TARGET_DATA_DIR = os.path.join(HOME_DIR, '.cache/huggingface/lerobot/rafli/drone_roblox')

# --- PERBAIKAN: Path Assets sekarang berada di dalam KAGGLE_BASE ---
# Kita hapus '/2500' karena sistem mencari langsung di bawah rafli/drone_roblox
FINAL_TARGET_NS = os.path.join(KAGGLE_BASE, 'assets', 'rafli/drone_roblox')

# 1. Siapkan OpenPI (Install sebagai modul)
print('🚀 1/3: Menyambungkan kode OpenPI ke sistem...')
OPENPI_SRC = os.path.join(KAGGLE_BASE, 'openpi')
if os.path.exists(OPENPI_SRC):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', OPENPI_SRC, '--no-deps'])
    print('✅ OpenPI terpasang.')
else:
    print('❌ ERROR: Folder openpi tidak ditemukan di dalam repo!')

# 2. Siapkan Dataset LeRobot (Metode Link - Lebih Cepat)
print('\n🚀 2/3: Menyiapkan Dataset LeRobot...')
dataset_src = os.path.join(KAGGLE_BASE, 'drone_roblox')
if os.path.exists(dataset_src):
    os.makedirs(os.path.dirname(TARGET_DATA_DIR), exist_ok=True)
    if os.path.exists(TARGET_DATA_DIR):
        if os.path.islink(TARGET_DATA_DIR) or os.path.isdir(TARGET_DATA_DIR):
            try: shutil.rmtree(TARGET_DATA_DIR)
            except: os.unlink(TARGET_DATA_DIR)
    
    # Gunakan symlink untuk menghemat disk
    os.symlink(dataset_src, TARGET_DATA_DIR)
    print(f'✅ Dataset tersambung di: {TARGET_DATA_DIR}')
else:
    print('❌ ERROR: Folder drone_roblox tidak ditemukan!')

# 3. Siapkan Assets (Model & Norm Stats)
print('\n🚀 3/3: Menyiapkan Aset Model & Checkpoint...')
os.makedirs(FINAL_TARGET_NS, exist_ok=True)

# List file yang harus ada di dalam folder checkpoint
# Kita butuh norm_stats dan metadata di posisi 'assets'
required_assets = ['norm_stats.json', 'metadata.pt']

for file_name in required_assets:
    src_file = os.path.join(KAGGLE_BASE, file_name) # Sumber dari root repo
    dst_file = os.path.join(FINAL_TARGET_NS, file_name) # Target ke folder assets
    
    if os.path.exists(src_file):
        # Bersihkan file lama jika ada (untuk menghindari error 'File exists')
        if os.path.exists(dst_file): os.remove(dst_file)
        
        # Gunakan link untuk menghemat space
        try:
            os.link(src_file, dst_file)
            print(f'  🔗 Linked asset: {file_name}')
        except:
            shutil.copy2(src_file, dst_file)
            print(f'  📋 Copied asset: {file_name}')
    else:
        print(f'  ⚠️ WARNING: {file_name} tidak ditemukan di root repo!')

# Pastikan model.safetensors tetap bisa diakses di root (sudah otomatis ada di root)
print(f'\n🚀 Assets siap di: {FINAL_TARGET_NS}')
print(f'📊 Isi folder assets: {os.listdir(FINAL_TARGET_NS)}')
print('\n✨ PERSIAPAN SELESAI!')



In [ ]:
!pip install peft --quiet

In [ ]:
import os, torch, sys, shutil
import transformers, lerobot

# Ganti lokasi base ke folder download kita
OPENPI_ROOT = os.path.join(KAGGLE_BASE, 'openpi')

def patch_recursive(src, dst):
    """Menyisipkan file satu per satu tanpa menghapus folder asli sistem."""
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.isdir(s):
            if not os.path.exists(d): os.makedirs(d)
            patch_recursive(s, d)
        else:
            shutil.copy2(s, d)

print('🛠️ Memulai proses patching file (v3.0 - Compatibility Mode)...')

# 1. Patch OpenPI (Fix datetime.UTC)
download_py = os.path.join(OPENPI_ROOT, 'src/openpi/shared/download.py')
if os.path.exists(download_py):
    with open(download_py, 'r') as f:
        content = f.read().replace('datetime.UTC', 'datetime.timezone.utc')
    with open(download_py, 'w') as f:
        f.write(content)
    print('✅ Patch 1/4: OpenPI download.py OK.')
else:
    print('⚠️ WARNING: Patch 1 SKIP (File tidak ditemukan)')

# 2. Patch LeRobot v3.0 (Fix TypeError: stack())
lerobot_lib_path = os.path.dirname(lerobot.__file__)
# Perbaikan Path v3.0: Menghapus subfolder 'common'
lerobot_ds_py = os.path.join(lerobot_lib_path, 'datasets/lerobot_dataset.py')
if os.path.exists(lerobot_ds_py):
    with open(lerobot_ds_py, 'r') as f:
        content = f.read()
    replacements = [
        ('torch.stack(self.hf_dataset["timestamp"])', 'torch.stack(list(self.hf_dataset["timestamp"]))'),
        ('torch.stack(self.hf_dataset["episode_index"])', 'torch.stack(list(self.hf_dataset["episode_index"]))'),
        ('torch.stack(self.hf_dataset["index"])', 'torch.stack(list(self.hf_dataset["index"]))'),
        ('torch.stack(self.hf_dataset["task_index"])', 'torch.stack(list(self.hf_dataset["task_index"]))'),
        ('torch.stack(self.hf_dataset.select(q_idx)[key])', 'torch.stack(list(self.hf_dataset.select(q_idx)[key]))')
    ]
    for old, new in replacements:
        content = content.replace(old, new)
    with open(lerobot_ds_py, 'w') as f:
        f.write(content)
    print('✅ Patch 2/4: LeRobot v3.0 dataset.py OK.')
else:
    print('⚠️ WARNING: Patch 2 SKIP (File tidak ditemukan)')

# 3. Patch Transformers (OpenPI Custom Model Patch)
transformers_path = os.path.dirname(transformers.__file__)
patch_source = os.path.join(OPENPI_ROOT, 'src/openpi/models_pytorch/transformers_replace')
if os.path.exists(patch_source):
    patch_recursive(patch_source, transformers_path)
    print('✅ Patch 3/4: Transformers OpenPI-Files OK.')
else:
    print('⚠️ WARNING: Patch 3 SKIP (Folder patch tidak ditemukan)')

# 4. Patch Model Loading (KHUSUS UNTUK PI0 CLOUD INFERENCE)
model_py = os.path.join(OPENPI_ROOT, 'src/openpi/models/model.py')
if os.path.exists(model_py):
    with open(model_py, 'r') as f:
        content = f.read()
    
    # Logic untuk memuat bobot secara non-strict (agar LoRA dari checkpoint bisa masuk)
    lora_patch = """
        # --- PATCH LORA INFERENCE (MATCH FORCE RESUME) ---
        from safetensors.torch import load_file
        print(f"🚀 Memuat checkpoint taktis dari: {weight_path}")
        try:
            from peft import get_peft_model, LoraConfig
            print("   ↳ Menyuntikkan struktur LoRA (r=8)...")
            lora_config = LoraConfig(
                r=8, lora_alpha=16, target_modules="all-linear", bias="none"
            )
            model = get_peft_model(model, lora_config)
            
            print("   ↳ Memasukkan bobot ke model (strict=False)...")
            state_dict = load_file(weight_path)
            model.load_state_dict(state_dict, strict=False)
            print(f"✅ Bobot LoRA Berhasil Dimuat! (Keys matched: {len(state_dict)})")
        except Exception as e:
            print(f"❌ ERROR SAAT MUAT LORA: {e}")
        # -------------------------------------------------
    """
    
    if "PATCH LORA INFERENCE" not in content:
        content = content.replace("safetensors.torch.load_model(model, weight_path)", lora_patch)
        with open(model_py, 'w') as f:
            f.write(content)
        print('✅ Patch 4/4: Model.py LATEST-RESTORE OK.')
else:
    print('⚠️ WARNING: Patch 4 SKIP (File tidak ditemukan)')

print('-' * 50)
print('🚀 Patch Selesai! Kamu siap melakukan pengujian model Pi0.')



In [ ]:
# ============================================================
# Patch LeRobot Tambahan (Fix TypeError: stack() - double check)
# ============================================================
import os, lerobot

lerobot_lib_path = os.path.dirname(lerobot.__file__)
lerobot_file = os.path.join(lerobot_lib_path, 'common/datasets/lerobot_dataset.py')

print(f'🔍 Mencari file di: {lerobot_file}')

if os.path.exists(lerobot_file):
    with open(lerobot_file, 'r') as f:
        content = f.read()

    replacements = [
        ('torch.stack(self.hf_dataset["timestamp"])', 'torch.stack(list(self.hf_dataset["timestamp"]))'),
        ('torch.stack(self.hf_dataset["episode_index"])', 'torch.stack(list(self.hf_dataset["episode_index"]))'),
        ('torch.stack(self.hf_dataset["index"])', 'torch.stack(list(self.hf_dataset["index"]))'),
        ('torch.stack(self.hf_dataset["task_index"])', 'torch.stack(list(self.hf_dataset["task_index"]))'),
        ('torch.stack(self.hf_dataset.select(q_idx)[key])', 'torch.stack(list(self.hf_dataset.select(q_idx)[key]))')
    ]

    patched_count = 0
    for old, new in replacements:
        if old in content:
            content = content.replace(old, new)
            patched_count += 1
            print(f'✅ Patching: {old[:50]}...')

    if patched_count > 0:
        with open(lerobot_file, 'w') as f:
            f.write(content)
        print(f'\n🎉 Berhasil menerapkan {patched_count} patch!')
    else:
        print('\nℹ️  Semua patch sudah diterapkan sebelumnya.')
else:
    print(f'❌ ERROR: File LeRobot tidak ditemukan!')

In [ ]:
# ============================================================
# Pre-download PaliGemma Tokenizer (WAJIB agar model bisa dimuat)
# ============================================================
import os
import urllib.request

print('🛠️ Mempersiapkan PaliGemma Tokenizer secara manual...')

cache_dir = '/root/.cache/openpi/big_vision'
os.makedirs(cache_dir, exist_ok=True)

tokenizer_url  = 'https://storage.googleapis.com/big_vision/paligemma_tokenizer.model'
target_path    = os.path.join(cache_dir, 'paligemma_tokenizer.model')

if not os.path.exists(target_path):
    print(f'⬇️ Sedang mengunduh Tokenizer ke {target_path} (sekitar 4 MB)...')
    try:
        urllib.request.urlretrieve(tokenizer_url, target_path)
        print('✅ Unduhan Tokenizer BERHASIL!')
    except Exception as e:
        print(f'❌ ERROR saat mengunduh: {e}')
else:
    print('✅ Tokenizer sudah tersedia di direktori target.')

In [ ]:
# ============================================================
# Setup sys.path dan konfigurasi environment untuk inferensi
# ============================================================
import sys, os, torch

# Path harus mengarah ke dalam sub-folder dronepivla (hasil download Cell 1)
OPENPI_SRC_PATH     = os.path.join(KAGGLE_BASE, 'openpi/src')
OPENPI_CLIENT_PATH  = os.path.join(KAGGLE_BASE, 'openpi/packages/openpi-client/src')

# Masukkan ke sys.path agar bisa di-import
for p in [OPENPI_SRC_PATH, OPENPI_CLIENT_PATH]:
    if os.path.exists(p):
        if p not in sys.path:
            sys.path.insert(0, p)
    else:
        print(f'⚠️ WARNING: Path tidak ditemukan: {p}')

# Set Environment Variables
os.environ['PYTHONPATH']                 = f'{OPENPI_SRC_PATH}:{OPENPI_CLIENT_PATH}'
os.environ['HF_HUB_OFFLINE']             = '1'  # Kita sudah punya filenya secara lokal
os.environ['JAX_PLATFORMS']              = 'cpu'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF']    = 'expandable_segments:True'

# Checkpoint utama adalah root dari folder download kita (KAGGLE_BASE)
# karena di sanalah file model.safetensors berada.
CHECKPOINT_DIR = KAGGLE_BASE 

print('--- ⚙️ Konfigurasi Inferensi ---')
print(f'Base Directory:  {KAGGLE_BASE}')
print(f'Checkpoint Dir:  {CHECKPOINT_DIR}')

# Verifikasi keberadaan file kunci
model_path = os.path.join(CHECKPOINT_DIR, "model.safetensors")
stats_path = os.path.join(CHECKPOINT_DIR, "norm_stats.json")

print(f'✅ model.safetensors exists : {os.path.exists(model_path)}')
print(f'✅ norm_stats.json exists    : {os.path.exists(stats_path)}')

print(f'🚀 CUDA Available           : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# ============================================================
# MUAT MODEL PI0 DARI CHECKPOINT 2500 STEPS
# ============================================================
import torch

from openpi.training import config as _config
from openpi.policies import policy_config as _policy_config

torch.cuda.empty_cache()

# Muat konfigurasi training yang sama persis dengan waktu training
train_config = _config.get_config('pi0_drone_lite')

print(f'✅ Config "{train_config.name}" dimuat.')
print(f'   action_dim:     {train_config.model.action_dim}  (internal)')
print(f'   action_horizon: {train_config.model.action_horizon}')
print(f'   paligemma:      {train_config.model.paligemma_variant}')
print()

# create_trained_policy secara otomatis:
# 1. Mendeteksi model.safetensors -> muat sebagai PyTorch model
# 2. Muat norm_stats dari CHECKPOINT_DIR/assets/rafli/drone_roblox/
# 3. Membangun pipeline transforms (DroneInputs, Normalize, dll)
# 4. Return Policy object yang siap untuk .infer()

print(f'🚀 Memuat model dari: {CHECKPOINT_DIR}')
print('   (Proses ini bisa memakan waktu 2-5 menit pertama kali)\n')

policy = _policy_config.create_trained_policy(
    train_config,
    CHECKPOINT_DIR,
    pytorch_device='cuda' if torch.cuda.is_available() else 'cpu',
)

if torch.cuda.is_available():
    vram_used  = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\n📊 Penggunaan VRAM setelah load: {vram_used:.2f} GB / {vram_total:.1f} GB')

print('\n✅ Policy berhasil dimuat dan siap untuk inferensi!')

In [ ]:
TARGET_DATA_DIR = '/kaggle/input/datasets/sjankaczar/dronepivla/drone_roblox/drone_roblox'

In [ ]:
# ============================================================
# SIAPKAN OBSERVASI (Custom Image Mode)
# ============================================================
import os, io, json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Path gambar yang kamu inginkan (WhatsApp Image)
IMAGE_PATH = "/kaggle/input/datasets/sjankaczar/dwaddwad/WhatsApp Image 2026-04-14 at 10.27.37.jpeg"

# Instruksi untuk model (sesuaikan dengan isi gambar kamu)
instruction = "look at the red cube" 

print(f"🔍 Mencoba memuat gambar kustom: {IMAGE_PATH}")

try:
    if not os.path.exists(IMAGE_PATH):
        # Jalankan pengecekan folder jika file tidak ditemukan
        parent_dir = os.path.dirname(IMAGE_PATH)
        if os.path.exists(parent_dir):
            print(f"⚠️ Folder ditemukan tapi file tidak ada. Isi folder: {os.listdir(parent_dir)}")
        raise FileNotFoundError("Gagal menemukan file gambar yang ditentukan!")

    # 1. Buka dan Preprocess Gambar
    # Pi0 butuh resolusi 224x224 dalam format RGB
    img_pil = Image.open(IMAGE_PATH).convert('RGB').resize((224, 224))
    sample_image = np.array(img_pil)
    
    print(f"✅ BERHASIL! Gambar dimuat dan di-resize ke {sample_image.shape}")
    print(f"✅ Instruksi: \"{instruction}\"")

except Exception as e:
    print(f"❌ ERROR: {e}")
    print("⚠️ Menggunakan gambar dummy sebagai fallback...")
    sample_image = np.zeros((224, 224, 3), dtype=np.uint8)
    sample_image[70:154, 70:154] = [220, 50, 50] # Kotak merah tiruan
    instruction = "look at the red cube (fallback)"

# Visualisasi untuk verifikasi sebelum dikirim ke model
plt.figure(figsize=(6, 6), facecolor='#1a1a2e')
plt.imshow(sample_image)
plt.title(f'📥 Input Frame (Custom Image)\n"{instruction}"', color='white', pad=15)
plt.axis('off')
plt.show()

# Masukkan ke variabel 'observation' untuk dikonsumsi sel inferensi selanjutnya
observation = {
    'image': sample_image.astype(np.uint8), # Harus uint8 (0-255)
    'state': np.zeros(4, dtype=np.float32),  # Inisialisasi drone diam
    'task': instruction,
}

print("-" * 50)
print("🚀 Data observasi siap untuk dilakukan inferensi Pi0!")



In [ ]:
# ============================================================
# JALANKAN INFERENSI
# ============================================================

# Siapkan observation dalam format yang diharapkan DroneInputs transform:
#   - 'image'  : np.ndarray (H, W, 3) uint8 -> DroneInputs akan mengonversi ke range [-1, 1]
#   - 'state'  : np.ndarray float32 (4,)    -> kecepatan saat ini [vx, vy, vz, yaw]
#   - 'task'   : str                         -> instruksi bahasa manusia

observation = {
    'image': sample_image,                      # (224, 224, 3) uint8
    'state': np.zeros(4, dtype=np.float32),     # Drone idle, kecepatan = 0
    'task':  instruction,
}

print('📤 Mengirim observasi ke model...')
print(f'  Image shape:  {observation["image"].shape}')
print(f'  State values: {observation["state"]}')
print(f'  Task:         {observation["task"]}')
print()

# Jalankan inferensi
with torch.no_grad():
    result = policy.infer(observation)

print('✅ Inferensi berhasil!')

In [ ]:
# ============================================================
# ANALISIS HASIL PREDIKSI
# ============================================================
import numpy as np

actions = result['actions']  # Shape: (action_horizon, 4)

print('=' * 65)
print('  HASIL INFERENSI MODEL PI0 DRONE VLA (2500 STEPS)')
print('=' * 65)
print(f'  Actions shape:       {actions.shape}')

if actions.shape[-1] == 4:
    status = '✅ BENAR — 4D (Vx, Vy, Vz, Yaw)'
else:
    status = f'⚠️  Tidak terduga: {actions.shape[-1]}D'
    
print(f'  Dimensi per timestep: {actions.shape[-1]}  ← {status}')
print(f'  Horizon prediksi:    {actions.shape[0]} timestep')
print()
print(f'  {"Step":<6} {"Vx (Maju)":>14} {"Vy (Samping)":>14} {"Vz (Naik)": >14} {"Yaw (Rotasi)":>14}')
print(f'  {"-"*62}')

for i in range(actions.shape[0]):
    vx, vy, vz, yaw = actions[i]
    print(f'  {i:<6} {vx:>14.6f} {vy:>14.6f} {vz:>14.6f} {yaw:>14.6f}')

print(f'  {"-"*62}')
print()
print('📊 Statistik Rata-rata Prediksi:')
print(f'  Mean Vx   (Maju):        {actions[:, 0].mean():>10.6f}')
print(f'  Mean Vy   (Samping):     {actions[:, 1].mean():>10.6f}')
print(f'  Mean Vz   (Naik/Turun):  {actions[:, 2].mean():>10.6f}')
print(f'  Mean Yaw  (Rotasi):      {actions[:, 3].mean():>10.6f}')
print()
print('=' * 65)
if actions.shape[-1] == 4:
    print('✅ KESIMPULAN: Model berhasil menghasilkan 4-dimensional actions!')
    print('   Output order: [Vx, Vy, Vz, Yaw] sesuai dengan DroneOutputs')
else:
    print(f'⚠️  PERINGATAN: Output memiliki {actions.shape[-1]} dimensi (bukan 4).')
print('=' * 65)

In [ ]:
# ============================================================
# VISUALISASI PREDIKSI AKSI (Bonus)
# ============================================================
import matplotlib.pyplot as plt

timesteps = list(range(actions.shape[0]))
labels = ['Vx (Maju)', 'Vy (Samping)', 'Vz (Naik/Turun)', 'Yaw (Rotasi)']
colors = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12']

fig, axes = plt.subplots(2, 2, figsize=(14, 8), facecolor='#1a1a2e')
axes = axes.flatten()
fig.suptitle(
    f'Pi0 Drone VLA — Prediksi Aksi (Checkpoint 2500 Steps)\n'
    f'Instruksi: "{instruction}"',
    color='white', fontsize=13, fontweight='bold'
)

for i, (ax, label, color) in enumerate(zip(axes, labels, colors)):
    ax.set_facecolor('#16213e')
    ax.plot(timesteps, actions[:, i], color=color, linewidth=2.5, marker='o', markersize=5)
    ax.axhline(y=0, color='white', linestyle='--', alpha=0.3, linewidth=1)
    ax.fill_between(timesteps, actions[:, i], 0, alpha=0.2, color=color)
    ax.set_title(label, color='white', fontsize=11, pad=8)
    ax.set_xlabel('Timestep', color='#aaaaaa', fontsize=9)
    ax.set_ylabel('Kecepatan (normalized)', color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa')
    for spine in ['bottom', 'left']:
        ax.spines[spine].set_color('#333366')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.1, color='white')

plt.tight_layout(rect=[0, 0, 1, 0.93])
output_path = '/kaggle/working/action_prediction.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print(f'\n✅ Grafik disimpan ke: {output_path}')

In [ ]:
# ============================================================
# UJI BATCH: Beberapa Frame Sekaligus
# ============================================================
import glob

# Kumpulkan semua file gambar dari dataset cache
all_images = []
chunk_dir = os.path.join(TARGET_DATA_DIR, 'data', 'chunk-000')
if os.path.exists(chunk_dir):
    for ext in ['*.png', '*.jpg', '*.jpeg']:
        all_images.extend(sorted(glob.glob(os.path.join(chunk_dir, '**', ext), recursive=True)))

N_TEST = min(5, len(all_images))
print(f'Total gambar ditemukan: {len(all_images)}')
print(f'Menguji pada {N_TEST} frame pertama...\n')

if N_TEST == 0:
    print('Tidak ada gambar. Membuat 3 gambar dummy untuk uji batch...')
    dummy_images = [np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8) for _ in range(3)]
    test_source = [(f'dummy_frame_{i}', img) for i, img in enumerate(dummy_images)]
else:
    test_source = [(img_path, np.array(Image.open(img_path).convert('RGB').resize((224, 224)))) for img_path in all_images[:N_TEST]]

print(f'{"Frame":<30} {"Vx":>10} {"Vy":>10} {"Vz":>10} {"Yaw":>10}')
print('-' * 72)

for img_name_or_path, img in test_source:
    fname = os.path.basename(str(img_name_or_path)) if isinstance(img_name_or_path, str) else img_name_or_path
    obs = {
        'image': img,
        'state': np.zeros(4, dtype=np.float32),
        'task':  instruction,
    }
    with torch.no_grad():
        res = policy.infer(obs)
    first_action = res['actions'][0]  # Timestep pertama saja
    vx, vy, vz, yaw = first_action
    print(f'{str(fname):<30} {vx:>10.5f} {vy:>10.5f} {vz:>10.5f} {yaw:>10.5f}')

print('-' * 72)
print(f'\n✅ Batch test selesai pada {N_TEST} frame!')
print(f'   Semua output berhasil mengeluarkan 4 dimensi aksi.')